# Unit 02 - Confounding (Exercise)

**Atoms served:** `U02-A2`, `U02-A4`, `U10-A8`

**Estimated runtime:** ~25 seconds

**After this notebook you can:** reproduce a confounded naive estimate, compute an adjusted estimate, and demonstrate bias persistence as `n` grows.

## Without code

Use the demo notebook outputs or these targets:

1. Naive `ATE` should be **positive** (~0.02 to 0.05) with true `ATE = 0`. On a control conversion near 0.16 that is a fake relative lift of roughly 15%.
2. Adjusted `ATE` should be near **0** (|value| < 0.02).
3. For `n = 10000`, naive `ATE` still positive while `SE < 0.02`.

Answer the same questions in prose.

## 1. The question

Same checkout story as the demo. Your job: compute the naive gap, adjust for `paid_channel`, then show precision tightening around a biased value.

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

Data generation is provided. True `ATE = 0`.

In [ ]:
n = 4000
paid = np.random.binomial(1, 0.35, n)
D = np.random.binomial(1, 0.55 + 0.25 * paid)
base = 0.12 + 0.18 * paid
Y = np.random.binomial(1, np.clip(base, 0, 1))
df = pd.DataFrame({'D_i': D, 'paid_channel': paid, 'Y_i': Y})

## 4. The naive move

**TODO:** Compute the naive `ATE` as treatment mean minus control mean. Print it rounded to 4 decimals. Expect a small **positive** number with seed 42 - small in absolute terms, but large next to a true effect of exactly zero.

In [ ]:
# TODO: naive ATE
naive_ate = None
assert naive_ate is not None
print('Naive ATE:', round(naive_ate, 4))
assert naive_ate > 0.01, 'Expected a spurious positive lift; check your formula'

## 5. What actually happens

**TODO:** Compute the average of channel-specific `ATE`s (paid arm gap and organic arm gap). Expect near **0**.

In [ ]:
# TODO: adjusted ATE
adj_ate = None
assert adj_ate is not None
print('Adjusted ATE:', round(adj_ate, 4))
assert abs(adj_ate) < 0.02, 'Adjusted estimate should be near zero'

**TODO:** For `n = 10000`, compute naive `ATE` and its approximate `SE`. Assert `SE < 0.02` and `ATE > 0`.

In [ ]:
# TODO: large-n bias demo
n_big = 10000
# regenerate data with same DGP as above
big_ate = None
big_se = None
assert big_ate is not None and big_se is not None
print('n=10000 naive ATE:', round(big_ate, 4), 'SE:', round(big_se, 4))
assert big_se < 0.02
assert big_ate > 0

## 6. What you do about it

Write one sentence: why is a significant naive result still untrustworthy here?

---

**Takeaway:** Adjust or randomise; never treat precision as proof of causality.

**Back to the unit:** [V1](../V1/units/unit-02-causality-and-confounding/README.md) · [V2](../V2/units/unit-02-causality-and-confounding/README.md)

## Hints

- Naive `ATE`: `df.groupby('D_i')['Y_i'].mean()` then subtract.
- Adjusted: compute `ATE` within each `paid_channel` level, then average the two gaps.
- `SE` for a difference of proportions: `sqrt(p1(1-p1)/n1 + p0(1-p0)/n0)`.

## Spoiler - full solution

<details><summary>Click only after you tried</summary>

```python
naive_ate = df.groupby('D_i')['Y_i'].mean()[1] - df.groupby('D_i')['Y_i'].mean()[0]
by = df.groupby(['paid_channel','D_i'])['Y_i'].mean().unstack()
adj_ate = ((by.loc[1,1]-by.loc[1,0]) + (by.loc[0,1]-by.loc[0,0]))/2
paid_b = np.random.binomial(1, 0.35, n_big)
D_b = np.random.binomial(1, 0.55 + 0.25 * paid_b)
Y_b = np.random.binomial(1, np.clip(0.12 + 0.18*paid_b, 0, 1))
tmp = pd.DataFrame({'D_i': D_b, 'Y_i': Y_b})
m = tmp.groupby('D_i')['Y_i'].mean()
big_ate = m[1]-m[0]
big_se = np.sqrt(tmp[tmp.D_i==1].Y_i.var()/(tmp.D_i==1).sum() + tmp[tmp.D_i==0].Y_i.var()/(tmp.D_i==0).sum())
```

</details>